# Análise para Redistribuição de Bicicletas Compartilhadas em São Paulo

## Parte 3 do Case Quant

Objetivo: entender padrões de demanda, risco de escassez e lotação, e fundamentar uma política de até três movimentações por hora.

## Estrutura do documento

0. Inicialização e carregamento dos dados permitidos.  
1. Estrutura, cobertura e qualidade dos dados.  
2. Análise exploratória de demanda, sazonalidade e desequilíbrio.  
3. Diagnóstico operacional e sinais de risco.  
4. Evidências para a interface do operador.  
5. Perfil preditivo e validação fora da amostra.  
6. Políticas candidatas e função de perda.  
7. Experimentos no avaliador local e decisão.  
8. Retomada da Parte 1: evidências e respostas da interface.

# 0. Inicialização, dependências e carregamento

In [170]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # evita incompatibilidade entre matplotlib e o backend inline do ambiente
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

# Localiza dados/ independentemente do diretório em que o servidor Jupyter foi iniciado.
CANDIDATOS_DADOS = []
for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    CANDIDATOS_DADOS.extend((
        base / "dados",
        base / "case_quant" / "dados",
        base / "case_quant_candidato" / "case_quant" / "dados",
    ))
DADOS_DIR = next((caminho for caminho in CANDIDATOS_DADOS if caminho.is_dir()), None)
if DADOS_DIR is None:
    raise FileNotFoundError("Não foi possível localizar a pasta dados/. Diretório atual: " + str(Path.cwd()))

print(f"Python: {sys.version.split()[0]}")
print(f"Dados: {DADOS_DIR.resolve()}")

Python: 3.12.3
Dados: /home/inteli/Downloads/case_quant_candidato/case_quant/dados


In [171]:
# Somente estes arquivos são autorizados para aprendizado pela solução final.
# Arquivos de simulação e *_baseline.csv ficam deliberadamente fora desta célula.
ARQUIVOS_PERMITIDOS = {
    "estacoes": ("estacoes.csv", []),
    "viagens": ("viagens.csv", ["start_time", "end_time"]),
    "historico_status": ("historico_status.csv", ["timestamp"]),
    "clima": ("clima.csv", ["timestamp"]),
    "calendario": ("calendario.csv", ["date"]),
    "snapshot_atual": ("snapshot_atual.csv", ["last_update"]),
}

dados_permitidos = {
    nome: pd.read_csv(DADOS_DIR / arquivo, parse_dates=colunas_data)
    for nome, (arquivo, colunas_data) in ARQUIVOS_PERMITIDOS.items()
}

estacoes = dados_permitidos["estacoes"]
viagens = dados_permitidos["viagens"]
historico_status = dados_permitidos["historico_status"]
clima = dados_permitidos["clima"]
calendario = dados_permitidos["calendario"]
snapshot_atual = dados_permitidos["snapshot_atual"]

resumo_carga = pd.DataFrame({
    "arquivo": [arquivo for arquivo, _ in ARQUIVOS_PERMITIDOS.values()],
    "linhas": [len(df) for df in dados_permitidos.values()],
    "colunas": [df.shape[1] for df in dados_permitidos.values()],
}, index=dados_permitidos.keys())
resumo_carga.index.name = "dataframe"
resumo_carga

,arquivo,linhas,colunas
dataframe,,,
estacoes,estacoes.csv,80,10
viagens,viagens.csv,165472,7
historico_status,historico_status.csv,706560,5
clima,clima.csv,2208,8
calendario,calendario.csv,153,8
snapshot_atual,snapshot_atual.csv,80,10


# 1. Estrutura e qualidade dos dados

## 1.1 Estações

In [172]:
estacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   station_id         80 non-null     object 
 1   name               80 non-null     object 
 2   address            80 non-null     object 
 3   neighborhood       80 non-null     object 
 4   zone               80 non-null     object 
 5   latitude           80 non-null     float64
 6   longitude          80 non-null     float64
 7   capacity           80 non-null     int64  
 8   installation_date  80 non-null     object 
 9   station_type       80 non-null     object 
dtypes: float64(2), int64(1), object(7)
memory usage: 6.4+ KB


### Qualidade cadastral

In [173]:
pd.Series({
    "linhas": len(estacoes),
    "station_id únicos": estacoes["station_id"].nunique(),
    "linhas duplicadas": estacoes.duplicated().sum(),
    "valores ausentes": int(estacoes.isna().sum().sum()),
}, name="auditoria_estacoes")

linhas               80
station_id únicos    80
linhas duplicadas     0
valores ausentes      0
Name: auditoria_estacoes, dtype: int64

### Capacidade por tipo de estação

In [174]:
estacoes.groupby("station_type")["capacity"].agg(["count", "min", "mean", "max"]).round(1)

,count,min,mean,max
station_type,,,,
express,12,12,14.100,17
premium,20,20,24.200,30
standard,48,15,20.200,24


## 1.2 Viagens

In [175]:
viagens.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165472 entries, 0 to 165471
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   trip_id           165472 non-null  object        
 1   start_time        165472 non-null  datetime64[ns]
 2   end_time          165472 non-null  datetime64[ns]
 3   start_station_id  165472 non-null  object        
 4   end_station_id    165472 non-null  object        
 5   duration_minutes  165472 non-null  float64       
 6   user_type         165472 non-null  object        
dtypes: datetime64[ns](2), float64(1), object(4)
memory usage: 8.8+ MB


### Integridade temporal e referencial

In [176]:
ids_estacoes = set(estacoes["station_id"])
pd.Series({
    "trip_id duplicados": viagens["trip_id"].duplicated().sum(),
    "origens desconhecidas": (~viagens["start_station_id"].isin(ids_estacoes)).sum(),
    "destinos desconhecidos": (~viagens["end_station_id"].isin(ids_estacoes)).sum(),
    "fim anterior ao início": (viagens["end_time"] < viagens["start_time"]).sum(),
}, name="auditoria_viagens")

trip_id duplicados        0
origens desconhecidas     0
destinos desconhecidos    0
fim anterior ao início    0
Name: auditoria_viagens, dtype: int64

### Duração e cruzamento de hora

In [177]:
viagens.assign(cruza_hora=viagens["start_time"].dt.floor("h") != viagens["end_time"].dt.floor("h")).groupby("user_type").agg(
    viagens=("trip_id", "size"),
    duracao_mediana_min=("duration_minutes", "median"),
    duracao_p95_min=("duration_minutes", lambda s: s.quantile(.95)),
    proporcao_cruza_hora=("cruza_hora", "mean"),
).round(3)

,viagens,duracao_mediana_min,duracao_p95_min,proporcao_cruza_hora
user_type,,,,
casual,41237,28.000,67.800,0.505
subscriber,124235,28.200,69.000,0.510


## 1.3 Histórico de status

In [178]:
historico_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 706560 entries, 0 to 706559
Data columns (total 5 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   timestamp        706560 non-null  datetime64[ns]
 1   station_id       706560 non-null  object        
 2   available_bikes  706560 non-null  int64         
 3   available_docks  706560 non-null  int64         
 4   status           706560 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 27.0+ MB


### Cobertura da série de status

In [179]:
pd.Series({
    "início": historico_status["timestamp"].min(),
    "fim": historico_status["timestamp"].max(),
    "estações cobertas": historico_status["station_id"].nunique(),
    "pares timestamp-estação duplicados": historico_status.duplicated(["timestamp", "station_id"]).sum(),
}, name="cobertura_status")

início                                2024-07-01 00:00:00
fim                                   2024-09-30 23:45:00
estações cobertas                                      80
pares timestamp-estação duplicados                      0
Name: cobertura_status, dtype: object

### Invariante físico: bicicletas + vagas = capacidade

In [180]:
status_capacidade = historico_status.merge(estacoes[["station_id", "capacity"]], on="station_id", how="left")
pd.Series({
    "IDs sem capacidade cadastrada": status_capacidade["capacity"].isna().sum(),
    "leituras que violam o invariante": (status_capacidade["available_bikes"] + status_capacidade["available_docks"] != status_capacidade["capacity"]).sum(),
}, name="invariante_status")

IDs sem capacidade cadastrada       0
leituras que violam o invariante    0
Name: invariante_status, dtype: int64

## 1.4 Clima

In [181]:
clima.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   timestamp          2208 non-null   datetime64[ns]
 1   temperature_c      2208 non-null   float64       
 2   feels_like_c       2208 non-null   float64       
 3   humidity_pct       2208 non-null   float64       
 4   precipitation_mm   2208 non-null   float64       
 5   wind_speed_kmh     2208 non-null   float64       
 6   weather_condition  2208 non-null   object        
 7   uv_index           2208 non-null   int64         
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 138.1+ KB


### Cobertura horária

In [182]:
horas_esperadas = pd.date_range(clima["timestamp"].min(), clima["timestamp"].max(), freq="h")
pd.Series({
    "timestamps duplicados": clima["timestamp"].duplicated().sum(),
    "horas ausentes": len(horas_esperadas.difference(clima["timestamp"])),
    "valores ausentes": int(clima.isna().sum().sum()),
}, name="cobertura_clima")

timestamps duplicados    0
horas ausentes           0
valores ausentes         0
Name: cobertura_clima, dtype: int64

### Frequência de condições meteorológicas

In [183]:
clima["weather_condition"].value_counts().to_frame("horas")

,horas
weather_condition,
clear,1157
cloudy,977
rain,65
storm,9


## 1.5 Calendário

In [184]:
calendario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             153 non-null    datetime64[ns]
 1   day_of_week      153 non-null    int64         
 2   day_name         153 non-null    object        
 3   is_weekend       153 non-null    bool          
 4   is_holiday       153 non-null    bool          
 5   holiday_name     6 non-null      object        
 6   school_vacation  153 non-null    bool          
 7   special_event    12 non-null     object        
dtypes: bool(3), datetime64[ns](1), int64(1), object(3)
memory usage: 6.6+ KB


### Cobertura e consistência de datas

In [185]:
pd.Series({
    "datas duplicadas": calendario["date"].duplicated().sum(),
    "dia da semana inconsistente": (calendario["date"].dt.dayofweek != calendario["day_of_week"]).sum(),
    "feriados": calendario["is_holiday"].sum(),
    "dias com evento": calendario["special_event"].notna().sum(),
}, name="auditoria_calendario")

datas duplicadas                0
dia da semana inconsistente     0
feriados                        6
dias com evento                12
Name: auditoria_calendario, dtype: int64

## 1.6 Snapshot atual

In [186]:
snapshot_atual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   station_id       80 non-null     object        
 1   name             80 non-null     object        
 2   neighborhood     80 non-null     object        
 3   zone             80 non-null     object        
 4   capacity         80 non-null     int64         
 5   available_bikes  80 non-null     int64         
 6   available_docks  80 non-null     int64         
 7   status           80 non-null     object        
 8   last_update      80 non-null     datetime64[ns]
 9   bikes_pct        80 non-null     float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(5)
memory usage: 6.4+ KB


### Integridade do estado atual

In [187]:
pd.Series({
    "timestamp único": snapshot_atual["last_update"].nunique(),
    "invariantes bikes + docks ≠ capacidade": (snapshot_atual["available_bikes"] + snapshot_atual["available_docks"] != snapshot_atual["capacity"]).sum(),
    "estações fora de operação": (~snapshot_atual["status"].eq("active")).sum(),
}, name="integridade_snapshot")

timestamp único                           1
invariantes bikes + docks ≠ capacidade    0
estações fora de operação                 3
Name: integridade_snapshot, dtype: int64

### Estações críticas no snapshot - menos que 10% ou mais que 90% de ocupação

In [188]:
snapshot_atual.assign(ocupacao=snapshot_atual["available_bikes"] / snapshot_atual["capacity"]).query("status == 'active' and (ocupacao <= .10 or ocupacao >= .90)")[
    ["station_id", "name", "zone", "available_bikes", "available_docks", "capacity", "ocupacao"]
].sort_values(["ocupacao", "station_id"])

,station_id,name,zone,available_bikes,available_docks,capacity,ocupacao
9,ST010,Mercado Municipal,Centro,0,12,12,0.000
32,ST033,Vila Clementino,Zona Sul,0,17,17,0.000
59,ST060,Jaçanã,Zona Norte,0,14,14,0.000
60,ST061,Mandaqui,Zona Norte,0,23,23,0.000
65,ST066,Parada Inglesa,Zona Norte,0,20,20,0.000
70,ST071,Belém,Zona Leste,0,23,23,0.000
74,ST075,Parque do Carmo,Zona Leste,0,22,22,0.000
76,ST077,São Mateus,Zona Leste,0,16,16,0.000
56,ST057,Freguesia do Ó,Zona Norte,1,22,23,0.043
30,ST031,Museu do Ipiranga,Zona Sul,1,21,22,0.045


### Objetivo da seção: entender padrões de demanda antes de modelar risco.

# 2. Demanda, sazonalidade e desequilíbrio

Análise exploratória com dados históricos permitidos. Viagens realizadas podem subestimar demanda quando há escassez ou lotação.

## 2.1 Retiradas por hora e dia da semana

**Como ler:** cada célula é o total de viagens iniciadas no período histórico para aquela combinação de dia da semana e hora. Ela mostra quando a rede concentra demanda; não é uma média diária.

In [189]:
viagens.assign(
    hora=viagens["start_time"].dt.hour,
    dia_semana=viagens["start_time"].dt.dayofweek,
).groupby(["dia_semana", "hora"]).size().unstack("hora", fill_value=0).rename(
    index={0: "Seg", 1: "Ter", 2: "Qua", 3: "Qui", 4: "Sex", 5: "Sáb", 6: "Dom"}
)

hora,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
dia_semana,,,,,,,,,,,,,,,,,,,,,,,,
Seg,194,115,73,67,133,456,1237,2264,2493,1708,1083,1220,1438,1346,1232,1423,1805,2760,3035,2129,1188,792,515,287
Ter,172,104,74,70,102,368,1077,2003,2192,1614,1036,1113,1292,1253,1205,1443,1782,2636,2831,1766,1039,772,463,229
Qua,156,80,57,74,99,366,1146,2124,2349,1521,992,1060,1231,1231,1184,1209,1635,2552,2832,2000,1178,703,448,269
Qui,162,95,56,63,121,354,1049,2024,2408,1580,1039,1101,1195,1233,1097,1265,1697,2288,2684,1738,1195,825,495,256
Sex,162,99,70,58,96,400,1160,2247,2504,1588,1005,1043,1275,1229,1195,1375,1827,2912,2946,1997,1185,714,495,224
Sáb,133,74,55,35,38,79,154,315,531,822,1123,1316,1535,1538,1457,1371,1237,922,630,496,299,273,187,144
Dom,135,85,50,29,41,67,159,343,527,828,1114,1284,1413,1431,1407,1313,1237,1011,755,524,366,276,219,143


## 2.2 Saldo de viagens por estação e hora

**Como ler:** `hora` é a hora do evento (0–23). `retiradas` conta viagens que começam na estação; `devolucoes`, viagens que terminam nela. `saldo = devolucoes − retiradas`, acumulado no histórico para aquela estação-hora. Saldo negativo indica pressão para esvaziar; positivo, pressão para lotar. Não é o estoque real nem uma previsão isolada.

In [190]:
retiradas_hora = viagens.assign(hora=viagens["start_time"].dt.hour).groupby(
    ["start_station_id", "hora"]
).size().rename("retiradas")
devolucoes_hora = viagens.assign(hora=viagens["end_time"].dt.hour).groupby(
    ["end_station_id", "hora"]
).size().rename("devolucoes")
fluxo_estacao_hora = pd.concat([retiradas_hora, devolucoes_hora], axis=1).fillna(0).astype(int)
fluxo_estacao_hora.index = fluxo_estacao_hora.index.set_names(["station_id", "hora"])
fluxo_estacao_hora["saldo"] = fluxo_estacao_hora["devolucoes"] - fluxo_estacao_hora["retiradas"]
n_dias_historico = viagens["start_time"].dt.normalize().nunique()
fluxo_estacao_hora["saldo_medio_diario"] = fluxo_estacao_hora["saldo"] / n_dias_historico
fluxo_estacao_hora.sort_values("saldo").head(10)

retiradas  devolucoes  saldo  saldo_medio_diario
station_id hora                                                  
ST035      8           508         193   -315              -3.424
ST037      8           498         194   -304              -3.304
ST035      7           432         143   -289              -3.141
ST009      8           526         253   -273              -2.967
           7           454         204   -250              -2.717
ST037      7           406         158   -248              -2.696
ST008      8           334          91   -243              -2.641
ST030      7           384         156   -228              -2.478
ST002      7           468         244   -224              -2.435
ST019      18          427         207   -220              -2.391

## 2.3 Viagens e temperatura

**Como ler:** as tabelas abaixo comparam a média de retiradas por hora observada em cada faixa de temperatura. São associações descritivas; temperatura também varia com horário, estação do ano e chuva.

In [191]:
demanda_horaria = viagens.groupby(
    viagens["start_time"].dt.floor("h")
).size().rename("retiradas").reset_index().rename(columns={"start_time": "timestamp"})
demanda_contexto = demanda_horaria.merge(clima, on="timestamp", how="left")
demanda_contexto["faixa_temperatura"] = pd.cut(
    demanda_contexto["temperature_c"],
    bins=[-np.inf, 15, 20, 24, 28, 32, np.inf],
    labels=["<15°C", "15–20°C", "20–24°C", "24–28°C", "28–32°C", ">32°C"],
)
demanda_contexto.head()

,timestamp,retiradas,temperature_c,feels_like_c,humidity_pct,precipitation_mm,wind_speed_kmh,weather_condition,uv_index,faixa_temperatura
0,2024-07-01 00:00:00,6,20.800,21.000,70.000,0.000,4.400,cloudy,0,20–24°C
1,2024-07-01 01:00:00,3,19.700,19.600,63.300,0.000,5.700,clear,0,15–20°C
2,2024-07-01 02:00:00,5,18.800,19.100,73.500,0.000,7.200,cloudy,0,15–20°C
3,2024-07-01 03:00:00,6,18.400,18.600,67.000,0.000,7.000,cloudy,0,15–20°C
4,2024-07-01 04:00:00,6,18.400,19.200,68.300,0.000,6.000,cloudy,0,15–20°C


## 2.4 Demanda horária condicionada à temperatura

**Como ler:** cada célula compara o volume médio de retiradas entre faixas de temperatura, dentro da mesma hora. Assim reduzimos a confusão entre temperatura e pico de demanda, mas ainda não inferimos causalidade.

In [192]:
demanda_contexto.assign(
    hora=demanda_contexto["timestamp"].dt.hour
).groupby(
    ["hora", "faixa_temperatura"], observed=False
)["retiradas"].mean().unstack("faixa_temperatura").round(1)

faixa_temperatura,<15°C,15–20°C,20–24°C,24–28°C,28–32°C,>32°C
hora,,,,,,
0,9.000,12.300,11.600,15.000,NaN,NaN
1,5.600,7.500,6.800,7.300,NaN,NaN
2,3.500,5.000,5.100,2.000,NaN,NaN
3,4.600,4.300,4.700,NaN,NaN,NaN
4,6.600,7.000,8.000,NaN,NaN,NaN
5,21.100,23.100,24.000,NaN,NaN,NaN
6,69.400,65.100,55.000,NaN,NaN,NaN
7,117.600,130.600,102.200,41.000,NaN,NaN
8,147.800,141.100,143.600,39.000,NaN,NaN


## 2.5 Exposição histórica a ocupação crítica

**Como ler:** cada proporção é a fração de leituras de 15 minutos em que estações ativas da zona estavam quase vazias (≤10%) ou quase cheias (≥90%). Ela mede recorrência histórica de extremos, não a probabilidade direta de falha na próxima hora.

In [193]:
ocupacao_historica = historico_status.merge(
    estacoes[["station_id", "zone", "capacity"]], on="station_id", how="left"
)
ocupacao_historica["ocupacao"] = (
    ocupacao_historica["available_bikes"] / ocupacao_historica["capacity"]
)
ocupacao_historica["critica_baixa"] = ocupacao_historica["ocupacao"] <= .10
ocupacao_historica["critica_alta"] = ocupacao_historica["ocupacao"] >= .90
(100 * ocupacao_historica.query("status == 'active'").groupby("zone")[
    ["critica_baixa", "critica_alta"]
].mean()).rename(columns={
    "critica_baixa": "% leituras em ≤10%",
    "critica_alta": "% leituras em ≥90%",
}).round(1)

,% leituras em ≤10%,% leituras em ≥90%
zone,,
Centro,2.300,36.800
Zona Leste,23.200,0.900
Zona Norte,21.100,0.800
Zona Oeste,9.200,7.000
Zona Sul,10.500,4.900


In [194]:
risco_historico = ocupacao_historica.query("status == 'active'").groupby("station_id").agg(
    proporcao_baixa=("critica_baixa", "mean"),
    proporcao_alta=("critica_alta", "mean"),
    ocupacao_media=("ocupacao", "mean"),
)
saldo_medio = fluxo_estacao_hora.groupby(level="station_id")["saldo_medio_diario"].mean()
estacoes.set_index("station_id")[["name", "zone"]].join(risco_historico).join(
    saldo_medio.rename("saldo_medio_diario_por_hora")
).assign(
    pct_leituras_baixas=lambda x: 100 * x["proporcao_baixa"],
    pct_leituras_altas=lambda x: 100 * x["proporcao_alta"],
).drop(columns=["proporcao_baixa", "proporcao_alta"]).sort_values(
    ["pct_leituras_baixas", "pct_leituras_altas"], ascending=False
).head(15).round(2)

,name,zone,ocupacao_media,saldo_medio_diario_por_hora,pct_leituras_baixas,pct_leituras_altas
station_id,,,,,,
ST075,Parque do Carmo,Zona Leste,0.190,-0.250,41.730,0.000
ST077,São Mateus,Zona Leste,0.190,-0.110,41.120,0.000
ST078,Guaianases,Zona Leste,0.200,-0.120,40.740,0.000
ST062,Horto Florestal,Zona Norte,0.210,-0.340,39.540,0.000
ST060,Jaçanã,Zona Norte,0.230,-0.190,36.880,0.000
ST063,Vila Maria,Zona Norte,0.270,-0.060,29.920,0.210
ST057,Freguesia do Ó,Zona Norte,0.230,-0.240,29.740,0.000
ST055,Estação Tucuruvi,Zona Norte,0.270,-0.200,27.610,0.080
ST073,Estação Penha,Zona Leste,0.250,-0.170,27.230,0.000


# 3. Diagnóstico operacional: risco de estoque e perdas potenciais

Esta seção não estima falhas diretamente: em julho–setembro observamos viagens realizadas, não intenções frustradas. Ela constrói sinais históricos de risco que posteriormente alimentarão uma função de perda.

## 3.1 Ocupação crítica por estação

**Como ler:** os percentuais são a fração de leituras de 15 minutos em que a estação estava ativa e em extremo de ocupação. Quanto maior o percentual, mais recorrente é o risco estrutural.

In [195]:
risco_por_estacao = ocupacao_historica.query("status == 'active'").groupby("station_id").agg(
    pct_baixa=("critica_baixa", lambda s: 100 * s.mean()),
    pct_alta=("critica_alta", lambda s: 100 * s.mean()),
    ocupacao_media=("ocupacao", "mean"),
    leituras=("ocupacao", "size"),
)
risco_por_estacao.join(
    estacoes.set_index("station_id")[["name", "zone", "capacity"]]
).sort_values(["pct_baixa", "pct_alta"], ascending=False).head(15).round(2)

,pct_baixa,pct_alta,ocupacao_media,leituras,name,zone,capacity
station_id,,,,,,,
ST075,41.730,0.000,0.190,8800,Parque do Carmo,Zona Leste,22
ST077,41.120,0.000,0.190,8676,São Mateus,Zona Leste,16
ST078,40.740,0.000,0.200,8732,Guaianases,Zona Leste,17
ST062,39.540,0.000,0.210,8732,Horto Florestal,Zona Norte,15
ST060,36.880,0.000,0.230,8752,Jaçanã,Zona Norte,14
ST063,29.920,0.210,0.270,8764,Vila Maria,Zona Norte,16
ST057,29.740,0.000,0.230,8736,Freguesia do Ó,Zona Norte,23
ST055,27.610,0.080,0.270,8728,Estação Tucuruvi,Zona Norte,22
ST073,27.230,0.000,0.250,8772,Estação Penha,Zona Leste,18


## 3.2 Criticidade ponderada pelo volume de viagens

**Como ler:** extremos de ocupação só são operacionalmente relevantes se coincidirem com uso. Esta tabela junta criticidade histórica e número de retiradas/devoluções realizadas; ela não mede demanda não atendida.

In [196]:
volume_estacao = pd.concat([
    viagens["start_station_id"].value_counts().rename("retiradas_realizadas"),
    viagens["end_station_id"].value_counts().rename("devolucoes_realizadas"),
], axis=1).fillna(0).astype(int)
risco_volume_estacao = estacoes.set_index("station_id")[["name", "zone", "capacity"]].join(
    risco_por_estacao
).join(volume_estacao)
risco_volume_estacao.sort_values(
    ["pct_baixa", "retiradas_realizadas"], ascending=False
).head(15).round(2)

,name,zone,capacity,pct_baixa,pct_alta,ocupacao_media,leituras,retiradas_realizadas,devolucoes_realizadas
station_id,,,,,,,,,
ST075,Parque do Carmo,Zona Leste,22,41.730,0.000,0.190,8800,1147,584
ST077,São Mateus,Zona Leste,16,41.120,0.000,0.190,8676,572,332
ST078,Guaianases,Zona Leste,17,40.740,0.000,0.200,8732,465,191
ST062,Horto Florestal,Zona Norte,15,39.540,0.000,0.210,8732,1232,474
ST060,Jaçanã,Zona Norte,14,36.880,0.000,0.230,8752,772,343
ST063,Vila Maria,Zona Norte,16,29.920,0.210,0.270,8764,767,640
ST057,Freguesia do Ó,Zona Norte,23,29.740,0.000,0.230,8736,1155,631
ST055,Estação Tucuruvi,Zona Norte,22,27.610,0.080,0.270,8728,1446,1002
ST073,Estação Penha,Zona Leste,18,27.230,0.000,0.250,8772,1189,813


## 3.3 Pressão de retirada e devolução por estação-hora

**Como ler:** a pressão normaliza o total histórico de viagens pela capacidade da estação. Pressão de retirada alta sugere necessidade de bicicletas; pressão de devolução alta sugere necessidade de vagas. Como é acumulada no histórico, serve para comparar padrões, não para quantificar risco de uma hora específica.

In [197]:
pressao_estacao_hora = fluxo_estacao_hora.reset_index().merge(
    estacoes[["station_id", "capacity", "name", "zone"]], on="station_id", how="left"
)
pressao_estacao_hora["pressao_retirada"] = (
    pressao_estacao_hora["retiradas"] / pressao_estacao_hora["capacity"]
)
pressao_estacao_hora["pressao_devolucao"] = (
    pressao_estacao_hora["devolucoes"] / pressao_estacao_hora["capacity"]
)
pressao_estacao_hora.sort_values("pressao_retirada", ascending=False)[
    ["station_id", "name", "zone", "hora", "retiradas", "capacity", "pressao_retirada"]
].head(15).round(2)

,station_id,name,zone,hora,retiradas,capacity,pressao_retirada
32,ST002,Estação República,Centro,8,522,20,26.100
200,ST009,Estação Consolação,Centro,8,526,21,25.050
8,ST001,Praça da Sé,Centro,8,509,21,24.240
31,ST002,Estação República,Centro,7,468,20,23.400
871,ST037,Estação Faria Lima,Zona Oeste,8,498,22,22.640
199,ST009,Estação Consolação,Centro,7,454,21,21.620
90,ST004,Estação Luz,Centro,18,453,22,20.590
7,ST001,Praça da Sé,Centro,7,432,21,20.570
824,ST035,Largo da Batata,Zona Oeste,8,508,26,19.540
473,ST020,Itaim Bibi,Zona Sul,17,410,21,19.520


## 3.4 Estrutura da futura função de perda

Para uma estação com (b) bicicletas, capacidade (C), retiradas previstas (R) e devoluções previstas (A), a aproximação inicial será:

\[
L(b) = \mathbb{E}[(R-b)_+] + \mathbb{E}[(A-(C-b))_+].
\]

O primeiro termo representa risco de não haver bicicleta; o segundo, risco de não haver vaga. A análise seguinte validará uma previsão de (R) e (A) fora da amostra antes de usar essa expressão para comparar transferências.

# 4. Evidências para responder às cinco perguntas da interface

O instante escolhido para a interface é o snapshot atual: 30/09/2024 14:30. Esta seção separa respostas observadas diretamente de estimativas ainda preliminares.

## Pergunta 1 — Qual é a situação atual da rede?

**Fonte:** snapshot atual. A tabela resume estoque, capacidade, criticidade e estações fora de operação. Esses números são observados, não previstos.

In [198]:
momento_interface = snapshot_atual["last_update"].iloc[0]
snapshot_interface = snapshot_atual.copy()
snapshot_interface["ocupacao"] = snapshot_interface["available_bikes"] / snapshot_interface["capacity"]
snapshot_interface["critica_baixa"] = snapshot_interface["status"].eq("active") & (snapshot_interface["ocupacao"] <= .10)
snapshot_interface["critica_alta"] = snapshot_interface["status"].eq("active") & (snapshot_interface["ocupacao"] >= .90)
pd.Series({
    "momento analisado": momento_interface,
    "bicicletas disponíveis": snapshot_interface["available_bikes"].sum(),
    "capacidade total": snapshot_interface["capacity"].sum(),
    "ocupação da rede (%)": round(100 * snapshot_interface["available_bikes"].sum() / snapshot_interface["capacity"].sum(), 1),
    "estações ativas": snapshot_interface["status"].eq("active").sum(),
    "estações fora de operação": (~snapshot_interface["status"].eq("active")).sum(),
    "críticas baixas ativas": snapshot_interface["critica_baixa"].sum(),
    "críticas altas ativas": snapshot_interface["critica_alta"].sum(),
})

momento analisado            2024-09-30 14:30:00
bicicletas disponíveis                       702
capacidade total                            1624
ocupação da rede (%)                      43.200
estações ativas                               77
estações fora de operação                      3
críticas baixas ativas                        13
críticas altas ativas                          9
dtype: object

In [199]:
snapshot_interface.groupby("zone").agg(
    estacoes=("station_id", "size"),
    bicicletas=("available_bikes", "sum"),
    capacidade=("capacity", "sum"),
    ocupacao_pct=("ocupacao", lambda s: 100 * s.mean()),
    criticas_baixas=("critica_baixa", "sum"),
    criticas_altas=("critica_alta", "sum"),
).round(1)

,estacoes,bicicletas,capacidade,ocupacao_pct,criticas_baixas,criticas_altas
zone,,,,,,
Centro,14,150,279,51.400,1,4
Zona Leste,14,99,277,35.600,3,1
Zona Norte,14,87,274,30.300,5,1
Zona Oeste,18,188,375,49.700,1,3
Zona Sul,20,178,419,39.300,3,0


## Pergunta 2 — Quais estações têm maior risco na próxima hora?

**Método preliminar:** média histórica de retiradas e devoluções para a mesma hora e dia da semana, comparada ao estoque e às vagas do snapshot. A perda estimada usa médias, portanto é uma triagem de risco — não uma probabilidade calibrada.

In [200]:
hora_referencia = momento_interface.hour
dia_semana_referencia = momento_interface.dayofweek
datas_referencia = viagens.loc[
    (viagens["start_time"].dt.hour == hora_referencia)
    & (viagens["start_time"].dt.dayofweek == dia_semana_referencia),
    "start_time",
].dt.normalize().nunique()
retiradas_ref = viagens.loc[
    (viagens["start_time"].dt.hour == hora_referencia)
    & (viagens["start_time"].dt.dayofweek == dia_semana_referencia)
].groupby("start_station_id").size().div(datas_referencia).rename("retiradas_previstas")
devolucoes_ref = viagens.loc[
    (viagens["end_time"].dt.hour == hora_referencia)
    & (viagens["end_time"].dt.dayofweek == dia_semana_referencia)
].groupby("end_station_id").size().div(datas_referencia).rename("devolucoes_previstas")
perfil_interface = snapshot_interface.set_index("station_id").join(retiradas_ref).join(devolucoes_ref)
perfil_interface[["retiradas_previstas", "devolucoes_previstas"]] = perfil_interface[
    ["retiradas_previstas", "devolucoes_previstas"]
].fillna(0)
perfil_interface[["retiradas_previstas", "devolucoes_previstas"]].head()

,retiradas_previstas,devolucoes_previstas
station_id,,
ST001,1.857,2.786
ST002,2.214,2.929
ST003,1.857,1.643
ST004,1.929,1.714
ST005,4.643,4.571


In [201]:
perfil_interface["risco_retirada"] = (perfil_interface["retiradas_previstas"] - perfil_interface["available_bikes"]).clip(lower=0)
perfil_interface["risco_devolucao"] = (perfil_interface["devolucoes_previstas"] - perfil_interface["available_docks"]).clip(lower=0)
perfil_interface["perda_esperada_aproximada"] = perfil_interface["risco_retirada"] + perfil_interface["risco_devolucao"]
perfil_interface.query("status == 'active'").sort_values(
    "perda_esperada_aproximada", ascending=False
)[["name", "zone", "available_bikes", "available_docks", "retiradas_previstas", "devolucoes_previstas", "risco_retirada", "risco_devolucao", "perda_esperada_aproximada"]].head(10).round(2)

,name,zone,available_bikes,available_docks,retiradas_previstas,devolucoes_previstas,risco_retirada,risco_devolucao,perda_esperada_aproximada
station_id,,,,,,,,,
ST007,Largo do Arouche,Centro,21,0,1.430,2.210,0.000,2.210,2.210
ST006,Praça da Liberdade,Centro,18,0,0.860,1.710,0.000,1.710,1.710
ST010,Mercado Municipal,Centro,0,12,1.640,2.500,1.640,0.000,1.640
ST043,Parque Villa-Lobos,Zona Oeste,1,21,2.640,1.930,1.640,0.000,1.640
ST075,Parque do Carmo,Zona Leste,0,22,0.930,0.430,0.930,0.000,0.930
ST011,Praça Roosevelt,Centro,15,1,1.860,1.860,0.000,0.860,0.860
ST066,Parada Inglesa,Zona Norte,0,20,0.860,0.210,0.860,0.000,0.860
ST031,Museu do Ipiranga,Zona Sul,1,21,1.790,0.930,0.790,0.000,0.790
ST033,Vila Clementino,Zona Sul,0,17,0.790,0.930,0.790,0.000,0.790


## Perguntas 3 e 4 — Quais três intervenções e por quê?

**Estado atual da análise:** já é possível listar estações candidatas a receber bicicletas e a liberar vagas. A escolha de pares origem–destino, quantidade e ordem ainda depende da função de perda calibrada e da busca conjunta de até três movimentos; a tabela abaixo não é uma recomendação final.

In [202]:
candidatos_receber = perfil_interface.query("status == 'active'").sort_values(
    "risco_retirada", ascending=False
)[["name", "zone", "available_bikes", "retiradas_previstas", "risco_retirada"]].head(5)
candidatos_aliviar = perfil_interface.query("status == 'active'").sort_values(
    "risco_devolucao", ascending=False
)[["name", "zone", "available_docks", "devolucoes_previstas", "risco_devolucao"]].head(5)
{"candidatas_a_receber": candidatos_receber, "candidatas_a_aliviar": candidatos_aliviar}

{'candidatas_a_receber':                           name        zone  available_bikes  \
 station_id                                                    
 ST010        Mercado Municipal      Centro                0   
 ST043       Parque Villa-Lobos  Zona Oeste                1   
 ST075          Parque do Carmo  Zona Leste                0   
 ST066           Parada Inglesa  Zona Norte                0   
 ST031        Museu do Ipiranga    Zona Sul                1   
 
             retiradas_previstas  risco_retirada  
 station_id                                       
 ST010                     1.643           1.643  
 ST043                     2.643           1.643  
 ST075                     0.929           0.929  
 ST066                     0.857           0.857  
 ST031                     1.786           0.786  ,
 'candidatas_a_aliviar':                              name        zone  available_docks  \
 station_id                                                       
 ST007    

## Pergunta 5 — Qual é a confiança nas previsões?

A interface final deverá divulgar uma métrica temporal fora da amostra, por exemplo MAE de retiradas e devoluções em setembro com treino até agosto. Essa validação ainda será criada na próxima seção; até lá, o risco exibido é explicitamente preliminar.

In [203]:
pd.DataFrame([
    ["1. Situação atual", "Snapshot atual", "Observada", "Pronta"],
    ["2. Risco de indisponibilidade", "Perfil histórico + snapshot", "Estimativa por médias", "Preliminar"],
    ["3. Três intervenções", "Função de perda + busca conjunta", "Decisão operacional", "Pendente"],
    ["4. Justificativa", "Risco marginal por movimento", "Explicação aproximada", "Pendente"],
    ["5. Confiança", "Validação temporal", "MAE/bias fora da amostra", "Pendente"],
], columns=["Pergunta", "Evidência necessária", "Tipo de resposta", "Estado"])

,Pergunta,Evidência necessária,Tipo de resposta,Estado
0,1. Situação atual,Snapshot atual,Observada,Pronta
1,2. Risco de indisponibilidade,Perfil histórico + snapshot,Estimativa por médias,Preliminar
2,3. Três intervenções,Função de perda + busca conjunta,Decisão operacional,Pendente
3,4. Justificativa,Risco marginal por movimento,Explicação aproximada,Pendente
4,5. Confiança,Validação temporal,MAE/bias fora da amostra,Pendente


# 5. Modelagem preditiva e validação temporal

O modelo candidato é uma tabela de médias de retiradas e devoluções por estação, hora e dia da semana. Ele pode ser construído com csv, dict e statistics.mean, sem bibliotecas externas. Treinamos em julho-agosto e avaliamos em setembro.

## 5.1 Separação cronológica e painel completo

**Como ler:** o painel inclui zeros: uma estação sem viagem em uma hora também é uma observação. Isso evita avaliar o modelo apenas onde já houve demanda.

In [204]:
inicio_validacao = pd.Timestamp("2024-09-01")
ids_estacoes = estacoes["station_id"].sort_values().tolist()
datas_treino = pd.date_range(viagens["start_time"].min().normalize(), inicio_validacao - pd.Timedelta(days=1), freq="D")
datas_validacao = pd.date_range(inicio_validacao, viagens["start_time"].max().normalize(), freq="D")

def painel_eventos(coluna_estacao, coluna_tempo, nome):
    eventos = viagens.assign(data=viagens[coluna_tempo].dt.normalize(), hora=viagens[coluna_tempo].dt.hour)
    contagens = eventos.groupby([coluna_estacao, "data", "hora"]).size().rename(nome).reset_index()
    contagens = contagens.rename(columns={coluna_estacao: "station_id"})
    return contagens

retiradas_diarias = painel_eventos("start_station_id", "start_time", "retiradas")
devolucoes_diarias = painel_eventos("end_station_id", "end_time", "devolucoes")
pd.Series({"dias de treino": len(datas_treino), "dias de validação": len(datas_validacao), "estações": len(ids_estacoes)})

dias de treino       62
dias de validação    30
estações             80
dtype: int64

## 5.2 Perfil estação × hora × dia da semana

O fallback por zona/global será comparado depois; primeiro, medimos o perfil mais específico para saber se a granularidade sustenta a previsão.

In [205]:
grade_treino = pd.MultiIndex.from_product([ids_estacoes, datas_treino, range(24)], names=["station_id", "data", "hora"]).to_frame(index=False)
grade_treino["dia_semana"] = grade_treino["data"].dt.dayofweek
treino = grade_treino.merge(retiradas_diarias, on=["station_id", "data", "hora"], how="left").merge(devolucoes_diarias, on=["station_id", "data", "hora"], how="left").fillna({"retiradas": 0, "devolucoes": 0})
perfil_treino = treino.groupby(["station_id", "hora", "dia_semana"])[["retiradas", "devolucoes"]].mean().rename(columns={"retiradas": "retiradas_previstas", "devolucoes": "devolucoes_previstas"}).reset_index()
perfil_treino.head()

,station_id,hora,dia_semana,retiradas_previstas,devolucoes_previstas
0,ST001,0,0,0.333,0.444
1,ST001,0,1,0.444,0.111
2,ST001,0,2,0.111,0.111
3,ST001,0,3,0.222,0.000
4,ST001,0,4,0.222,0.444


## 5.3 MAE e viés fora da amostra

**Como ler:** MAE mede erro absoluto médio em viagens por estação-hora; viés positivo indica superprevisão, negativo indica subprevisão. Retiradas e devoluções são avaliadas separadamente.

In [206]:
grade_validacao = pd.MultiIndex.from_product([ids_estacoes, datas_validacao, range(24)], names=["station_id", "data", "hora"]).to_frame(index=False)
grade_validacao["dia_semana"] = grade_validacao["data"].dt.dayofweek
validacao = grade_validacao.merge(retiradas_diarias, on=["station_id", "data", "hora"], how="left").merge(devolucoes_diarias, on=["station_id", "data", "hora"], how="left").fillna({"retiradas": 0, "devolucoes": 0})
validacao = validacao.merge(perfil_treino, on=["station_id", "hora", "dia_semana"], how="left").fillna({"retiradas_previstas": 0, "devolucoes_previstas": 0})

def metricas_previsao(real, previsto):
    erro = real - previsto
    return pd.Series({"MAE": erro.abs().mean(), "viés": erro.mean(), "RMSE": np.sqrt((erro ** 2).mean())})

pd.DataFrame({
    "retiradas": metricas_previsao(validacao["retiradas"], validacao["retiradas_previstas"]),
    "devolucoes": metricas_previsao(validacao["devolucoes"], validacao["devolucoes_previstas"]),
}).T.round(3)

,MAE,viés,RMSE
retiradas,0.699,0.017,1.061
devolucoes,0.708,0.017,1.125


# 6. Políticas candidatas: função de perda marginal e seleção de movimentos

A função abaixo é determinística e deliberadamente simples. Ela pode ser copiada para a solução final sem numpy ou pandas; a validação acima informa se os perfis que a alimentam têm qualidade suficiente.

## 6.1 Perda estimada para um estoque inicial

Para estoque b, a perda aproxima falta de bicicletas e falta de vagas usando as médias previstas. Ela é uma aproximação: não modela ainda ordem individual dos eventos, viagens em trânsito ou redirecionamento.

In [207]:
def perda_estimada(bicicletas, capacidade, retiradas_previstas, devolucoes_previstas):
    perda_retirada = max(retiradas_previstas - bicicletas, 0)
    perda_devolucao = max(devolucoes_previstas - (capacidade - bicicletas), 0)
    return perda_retirada + perda_devolucao

# A função usa apenas operações e max: é compatível com a biblioteca padrão.
perda_estimada(3, 20, 6, 4)

3

## 6.2 Benefício marginal de receber ou ceder uma bicicleta

**Como ler:** benefício positivo ao receber significa redução de perda ao acrescentar uma bicicleta. Impacto negativo ao ceder significa que retirar uma bicicleta reduz a perda, tipicamente por abrir uma vaga em estação sob risco de lotação.

In [208]:
marginais = perfil_interface.query("status == 'active'").copy()
marginais["perda_atual"] = marginais.apply(lambda x: perda_estimada(x.available_bikes, x.capacity, x.retiradas_previstas, x.devolucoes_previstas), axis=1)
marginais["beneficio_receber_1"] = marginais.apply(lambda x: x.perda_atual - perda_estimada(min(x.available_bikes + 1, x.capacity), x.capacity, x.retiradas_previstas, x.devolucoes_previstas), axis=1)
marginais["impacto_ceder_1"] = marginais.apply(lambda x: perda_estimada(max(x.available_bikes - 1, 0), x.capacity, x.retiradas_previstas, x.devolucoes_previstas) - x.perda_atual, axis=1)
marginais.sort_values("beneficio_receber_1", ascending=False)[["name", "zone", "available_bikes", "beneficio_receber_1", "impacto_ceder_1"]].head(10).round(3)

,name,zone,available_bikes,beneficio_receber_1,impacto_ceder_1
station_id,,,,,
ST010,Mercado Municipal,Centro,0,1.000,0.000
ST043,Parque Villa-Lobos,Zona Oeste,1,1.000,1.000
ST075,Parque do Carmo,Zona Leste,0,0.929,0.000
ST066,Parada Inglesa,Zona Norte,0,0.857,0.000
ST031,Museu do Ipiranga,Zona Sul,1,0.786,1.000
ST033,Vila Clementino,Zona Sul,0,0.786,0.000
ST071,Belém,Zona Leste,0,0.500,0.000
ST077,São Mateus,Zona Leste,0,0.214,0.000
ST060,Jaçanã,Zona Norte,0,0.143,0.000


## Próximo experimento

Com estes resultados, a próxima etapa é comparar no avaliador local: nenhuma movimentação, regra simples de ocupação e regra de perda marginal com recálculo após cada movimento. Os dados de outubro entram somente nesta comparação retrospectiva, nunca no cache ou na decisão online.

# 7. Experimentos no avaliador local e decisão

Esta seção registra experimentos reprodutíveis. O cenário de outubro é usado somente para avaliar políticas localmente; nenhuma política lê os arquivos de simulação, baseline ou intenções durante o preparo ou a decisão.

## 7.1 Desenho experimental

| Política | Regra | Dados usados em decisão | Propósito |
|---|---|---|---|
| P0 | Sem movimentação | Nenhum | Baseline imóvel |
| P1 | Faixas de ocupação | Estado recebido no JSON | Referência operacional simples |
| P2 | Perda marginal gulosa | Estado recebido + cache de viagens históricas | Testar se previsão e risco marginal melhoram a seleção |

P1 move de estações com ao menos 80% de ocupação para estações com no máximo 20%, preservando 60% na origem e buscando 40% no destino. P2 gera perfis de retiradas/devoluções por estação, hora e dia da semana no passo de preparo e seleciona sequencialmente até três transferências de ganho marginal positivo.

## 7.2 Resultados no cenário local de outubro

**Como ler:** o score é o número de intenções não atendidas; menor é melhor. O cenário é conhecido apenas para avaliação local, portanto estes valores não são estimativas do cenário oculto.

In [209]:
resultados_experimentos = pd.DataFrame([
    {"politica": "P0: baseline imóvel", "falhas": 29332, "viagens_salvas": 0, "melhora_pct": 0.0, "movimentacoes": 0, "bicicletas_movidas": 0, "ms_por_chamada": None},
    {"politica": "P1: ocupação", "falhas": 5372, "viagens_salvas": 23960, "melhora_pct": 81.7, "movimentacoes": 1394, "bicicletas_movidas": 8969, "ms_por_chamada": 72},
    {"politica": "P2: perda marginal", "falhas": 4748, "viagens_salvas": 24584, "melhora_pct": 83.8, "movimentacoes": 1366, "bicicletas_movidas": 13571, "ms_por_chamada": 611},
    {"politica": "P3: perda marginal otimizada", "falhas": 4477, "viagens_salvas": 24855, "melhora_pct": 84.7, "movimentacoes": 1347, "bicicletas_movidas": 13366, "ms_por_chamada": 146},
])
resultados_experimentos

,politica,falhas,viagens_salvas,melhora_pct,movimentacoes,bicicletas_movidas,ms_por_chamada
0,P0: baseline imóvel,29332,0,0.000,0,0,NaN
1,P1: ocupação,5372,23960,81.700,1394,8969,72.000
2,P2: perda marginal,4748,24584,83.800,1366,13571,611.000
3,P3: perda marginal otimizada,4477,24855,84.700,1347,13366,146.000


## 7.3 Decisão e próximos testes

P3 substitui P2 como política-base: preserva a mesma função de perda, mas explora sua separabilidade por origem, destino e quantidade. Ela salva 271 viagens a mais que P2, usa menos movimentos e reduz o tempo de 611 para 146 ms por chamada.

Antes da entrega, os próximos testes devem ser ablações: restringir candidatos, limitar quantidades, comparar um, dois e três movimentos e inspecionar falhas de retirada/devolução nos arquivos gerados por `--saida`. Uma alteração só deve permanecer se conservar ou aumentar o ganho de P3 no avaliador local e não usar informação proibida.

## 7.4 Próximos experimentos e hipóteses

P3 seleciona três movimentos com **guloso sequencial**: escolhe o melhor ganho imediato, atualiza o estado e repete. Isso não garante a melhor combinação conjunta, pois uma primeira escolha menos atraente pode liberar dois movimentos seguintes com ganho total maior.

- **E1 — beam search conjunto:** manter as duas melhores sequências parciais e expandi-las até três movimentos. Testa diretamente se o guloso é míope; limitar a dois estados e cinco candidatos por estado protege o tempo.
- **E2 — quantidades de quebra:** avaliar apenas quantidades em que a perda marginal muda. Permite considerar transferências maiores sem varrer todas as quantidades de 1 a 10.
- **E3 — horizonte de duas horas:** somar a perda atual e uma fração da perda da próxima hora, usando somente perfis históricos já permitidos. Testa antecipação de picos.
- **E4 — tendência recente:** ajustar previsões pelo sinal das oito leituras recebidas. Testa adaptação a desvios recentes, mas deve ser limitado para não confundir efeito das próprias transferências com demanda.

A prioridade é E1, pois responde à restrição central do case — escolher conjuntamente até três movimentações — sem mudar o modelo preditivo.

### Resultado de E1 — busca conjunta por beam search

O teste foi feito até 03/out, com o mesmo cache e função de perda de P3. P4 expandiu sequências de até três movimentos, mantendo os melhores estados parciais.

| Política | Falhas | Viagens salvas | Tempo por chamada | Decisão |
|---|---:|---:|---:|---|
| P3 guloso sequencial | 331 | 1.772 | 124 ms | Manter |
| P4 beam=2, 5 candidatos | 362 | 1.741 | 92 ms | Rejeitar |
| P4 beam=3, 8 candidatos | 362 | 1.741 | 99 ms | Rejeitar |

A busca conjunta reduziu o tempo, mas piorou o score em 31 viagens no recorte. A interpretação é que a função de perda determinística ainda é uma aproximação imperfeita do simulador; aprofundar a busca nessa mesma função não é justificado. P3 permanece como política-base.

### Resultado de E2 — remover limite fixo de quantidade

**Método:** P5 avaliou todas as quantidades inteiras fisicamente viáveis, até o estoque da origem e as vagas do destino. Assim, removeu o teto arbitrário de 10 bicicletas de P3.

| Política | Falhas até 03/out | Viagens salvas | Bicicletas movidas | Tempo por chamada | Decisão |
|---|---:|---:|---:|---:|---|
| P3, máximo experimental de 10 | 331 | 1.772 | 1.405 | 124 ms | Manter como referência |
| P5, máximo físico | 491 | 1.612 | 2.239 | 100 ms | Rejeitar |

P5 propôs transferências de 15–20 bicicletas no primeiro instante. A perda determinística não penaliza suficientemente o risco futuro de esvaziar a origem ou lotar o destino, por isso a liberdade física levou a movimentos agressivos e pior score. A próxima variante deve usar limites dinâmicos derivados de demanda prevista e incerteza, não uma constante fixa.

### Resultado de E2b — reservas dinâmicas por demanda e incerteza

**Método:** P6 substituiu o teto fixo por uma reserva mínima de bicicletas e vagas. A reserva é demanda prevista mais `sqrt(retiradas + devoluções)`, aproximação de um desvio padrão Poisson. Portanto, o limite varia por estação e hora.

| Política | Falhas até 03/out | Viagens salvas | Bicicletas movidas | Tempo por chamada | Decisão |
|---|---:|---:|---:|---:|---|
| P3 | 331 | 1.772 | 1.405 | 124 ms | Manter |
| P5, máximo físico | 491 | 1.612 | 2.239 | 100 ms | Rejeitar |
| P6, reserva dinâmica | 402 | 1.701 | 1.904 | 104 ms | Rejeitar |

A reserva dinâmica reduz a agressividade de P5, mas ainda permite transferências grandes e não supera P3. O resultado reforça que a perda de uma hora não representa totalmente o valor futuro do estoque; não basta trocar um teto fixo por uma fórmula de buffer sem validar o efeito no simulador.

### Resultado de E3 — horizonte de duas horas

**Método:** P7 preserva a perda e busca de P3, mas soma a perda prevista da hora atual à da próxima hora. O estoque da segunda hora é aproximado por `b - retiradas_previstas + devolucoes_previstas`, limitado à capacidade. A política é reotimizada a cada chamada; o segundo movimento futuro não é congelado. Retiradas noturnas foram zeradas no horizonte, conforme a regra do simulador.

| Política | Falhas em outubro | Viagens salvas | Melhora | Tempo por chamada | Decisão |
|---|---:|---:|---:|---:|---|
| P3, horizonte de uma hora | 4.477 | 24.855 | 84,7% | 146 ms | Referência |
| P7, horizonte de duas horas | 4.046 | 25.286 | 86,2% | 119 ms | Nova política-base |

P7 salvou 431 viagens a mais que P3 no cenário local completo. Como a comparação usa exatamente o mesmo modelo de demanda e a mesma busca, a evidência atribui o ganho à antecipação de uma hora. A aproximação continua limitada por não simular eventos individuais, viagens em trânsito e redirecionamentos; esses limites devem ser explicitados na entrega.

## 7.5 Diagnóstico dos erros residuais de P7

As saídas gravadas pelo avaliador local permitem separar erros associados a estoque daqueles causados por status exógeno. Essa análise usa outubro **somente como diagnóstico retrospectivo**; seus resultados não entram no cache ou na decisão de P7.

| Tipo de falha residual | Total | Estação ativa | Maintenance/inactive |
|---|---:|---:|---:|
| Retirada | 2.642 | 1.465 | 1.177 |
| Devolução | 1.404 | 353 | 1.051 |

Assim, 2.228 das 4.046 falhas residuais (55%) ocorreram em estações fora de operação e não são diretamente evitáveis sem previsão de status futuro — informação que a política não recebe. Entre as falhas potencialmente tratáveis, predominam retiradas em estações ativas. Os picos residuais são 08h, 17–18h e 09h; ST034, ST027, ST066, ST062, ST057, ST051 e ST064 concentram falhas ativas de retirada.

Esse diagnóstico explica por que aprofundar a busca de movimentos teve retorno limitado: o gargalo remanescente é, em grande parte, incerteza e indisponibilidade exógena, não apenas seleção de pares.

## 7.6 Próximo experimento — previsão robusta por quantis

**Hipótese:** a média histórica subestima picos de demanda nas horas e estações que ainda sofrem falhas de retirada. Substituir a média por quantis moderadamente altos de retiradas e devoluções deve preservar mais bicicletas e vagas nos pontos voláteis.

**Método:** manter a estrutura de P7 — horizonte de duas horas, busca rápida e mesmos dados permitidos — e trocar as médias dos perfis por quantis empíricos de contagens por estação, hora e dia da semana. Comparar, no mínimo, P50 (média/mediana de referência), P70 e P75. O percentil é calculável com listas ordenadas da biblioteca padrão.

**Critério de sucesso:** só manter um quantil se reduzir as falhas totais e, em particular, as falhas de retirada em estações ativas no avaliador local, sem aumento material do tempo por chamada. Deve-se verificar também se a proteção extra contra escassez não eleva excessivamente falhas de devolução.

### Resultado de E5 — perfis por quantis empíricos

**Método:** P8 mantém horizonte de duas horas e a busca de P7, trocando apenas a média por um quantil das contagens históricas, incluindo zeros, para cada estação–hora–dia da semana.

| Perfil | Falhas até 03/out | Viagens salvas | Tempo por chamada | Decisão |
|---|---:|---:|---:|---|
| P7, média histórica | 306 | 1.797 | 112 ms | Manter |
| P8, P50 | 384 | 1.719 | 124 ms | Rejeitar |
| P8, P70 | 354 | 1.749 | 128 ms | Rejeitar |
| P8, P75 | 354 | 1.749 | 107 ms | Rejeitar |

P70 e P75 coincidiram porque há aproximadamente 13 dias de treino por combinação de dia da semana; ambos selecionaram a mesma posição ordenada. Quantis mais altos reduziram parcialmente a piora de P50, mas nenhum superou a média. A hipótese de que picos residuais vinham apenas de subprevisão média foi rejeitada; P7 permanece como política-base.

## 7.7 Plano de ablações seguintes

As hipóteses abaixo serão testadas uma a uma, mantendo a busca, os dados permitidos e o horizonte de duas horas de P7 como controle. Primeiro será usado o recorte até 03/out; somente variantes que superarem P7 nesse filtro seguirão para outubro completo.

| ID | Hipótese | Mudança isolada | Evidência de sucesso |
|---|---|---|---|
| H1 | Perda empírica | Usar a distribuição histórica completa de contagens em vez de apenas a média | Menos falhas, sobretudo retirada ativa |
| H2 | Noite às 21h | Estender o horizonte da decisão das 21h até 05h, com apenas devoluções | Menos devoluções noturnas e no dia seguinte |
| H3 | Suavização por zona | Combinar perfil da estação com perfil da zona para reduzir ruído | Menor MAE em setembro e melhor score |
| H4 | Clima corrente | Ajustar demanda pela condição climática observada | Ganho de MAE e score sem leakage |
| H5 | Confiabilidade de status | Penalizar destinos historicamente instáveis quando ativos | Menos movimentos desperdiçados e falhas em paradas |

Nenhuma hipótese pode usar intenções, status de outubro, baseline ou qualquer arquivo de simulação durante o preparo ou a decisão.

## 7.8 Resultados das cinco hipóteses seguintes

O protocolo manteve P7 como controle e alterou apenas um componente por vez. Todas as políticas foram preparadas apenas com os CSVs históricos permitidos e avaliadas no recorte de 01 a 03/out (51 decisões). A coluna de tempo é a média por chamada; portanto, todas atendem com ampla folga ao limite de 30 segundos.

| Hipótese / política | Alteração isolada | Falhas | Viagens salvas | Tempo/chamada | Decisão |
|---|---|---:|---:|---:|---|
| Controle P7 | média histórica, horizonte de 2 h | 306 | 1.797 | 112 ms | Base |
| H1 / P9 | perda esperada de uma Poisson parametrizada pela média histórica | 321 | 1.782 | 141 ms | Rejeitar |
| H2 / P10 | às 21h, horizonte até 05h; demais horas, 2 h | 306 | 1.797 | 213 ms | Rejeitar |
| H3 / P11 | perfil local suavizado com o perfil da zona | 343 | 1.760 | 154 ms | Rejeitar |
| H4 / P12 | multiplicador de demanda da condição climática corrente | 330 | 1.773 | 204 ms | Rejeitar |
| H5 / P13 | risco da hora seguinte ponderado pela probabilidade histórica de a estação permanecer ativa | **304** | **1.799** | 222 ms | Prosseguir |

**Leitura dos resultados.** H1 aumentou a aversão a rupturas, mas não selecionou melhores pares. H2 não mudou as decisões e apenas elevou o custo computacional. H3 diluiu informação local relevante. H4 também piorou: o clima corrente é um sinal agregado e não foi suficiente para reposicionar corretamente bicicletas entre estações. Em contraste, H5 atacou diretamente uma fonte importante de erro residual: a indisponibilidade futura de estações.

### Validação mensal da variante aprovada

P13 foi então avaliada entre 01 e 31/out (527 decisões), antes de ser promovida. O resultado foi **3.951 falhas**, contra 4.046 de P7: 95 falhas a menos. Foram salvas **25.381 viagens** (86,5% das 29.332 falhas do baseline), contra 25.286 (86,2%) de P7. A média foi **159 ms por decisão**, cerca de 189 vezes abaixo do limite de 30 s. Assim, P13 torna-se a política candidata, enquanto as quatro demais permanecem registradas como ablações rejeitadas.

A ponderação de H5 não consulta o status futuro: no instante da decisão ela usa somente a frequência histórica de `active` da combinação estação–hora–dia da semana para descontar o risco projetado da próxima hora.

## 7.9 Extensões de confiabilidade e estado do estoque

### H6 — continuidade condicional de status

**Hipótese:** como a política vê que uma estação está ativa no instante atual, a probabilidade relevante para a segunda hora seria `P(active em t+1 | active em t, estação, hora, dia da semana)`, em vez da frequência marginal de atividade usada por P13.

**Resultado:** P14 obteve 312 falhas e 1.791 viagens salvas até 03/out, em 140 ms por chamada. Ficou pior que P7 (306) e P13 (304), portanto foi rejeitada sem avaliação mensal. A amostra por combinação estação–hora–dia ficou pequena para estimar a transição de modo mais útil do que a frequência marginal.

### H7 — demanda condicionada ao estoque inicial

**Hipótese:** retiradas e devoluções podem depender do estoque observado no começo da hora. P15 divide a ocupação relativa em quatro faixas e estima, para cada estação–hora–dia–faixa, as contagens médias históricas; quando não há observação, retorna ao perfil-base de P7.

**Resultado:** P15 teve 480 falhas e 1.623 viagens salvas até 03/out, em 248 ms por chamada. A piora forte rejeita a hipótese nesta formulação: segmentar aproximadamente 13 observações históricas por combinação em quatro faixas elevou demais a variância, além de misturar demanda latente com demanda já censurada por falta de bicicletas ou vagas.

**Decisão:** manter P13 como candidata. H6 e H7 são evidências úteis contra aumentar a granularidade do perfil sem uma quantidade substancialmente maior de histórico.

# 8. Retomada — respostas da interface da Parte 1

Os experimentos encerram a escolha provisória da política: P13 é a referência porque reduziu as falhas no recorte e em outubro completo, com 159 ms por decisão. Esta seção transforma as evidências anteriores em conteúdo verificável para a interface no snapshot de 30/09/2024 às 14:30.

A regra de apresentação é importante: **estado atual** vem de `snapshot_atual.csv`; **risco e benefício** são previsões a partir dos perfis históricos permitidos; **qualidade da previsão** vem da validação temporal de julho–setembro. Não apresentar o resultado retrospectivo de outubro como se estivesse disponível no snapshot.

## 8.1 As cinco perguntas e a evidência disponível

| Pergunta da interface | Resposta que a interface deve construir | Evidência já produzida | Regra de comunicação |
|---|---|---|---|
| 1. Como está a rede agora? | bicicletas e vagas totais; estações ativas/inativas; ocupação por zona; lista de estações críticas | Seções 1 e 2; `snapshot_atual.csv`; capacidades e faixas de ocupação | Mostrar observação, não previsão. Separar inativas das estações ativas críticas. |
| 2. O que pode acontecer na próxima hora? | risco de falta de bicicleta e de vaga por estação ativa; principais estações em risco | Perfil estação–hora–dia da semana, função de perda e validação da seção 5 | Chamar de estimativa; mostrar retirada e devolução separadamente. |
| 3. Quais são as até três melhores movimentações? | origem, destino, quantidade, ordem e estações envolvidas | Política P13: perda em duas horas e confiabilidade histórica de status | Mostrar zero, uma, duas ou três ações conforme ganho positivo; não preencher ações artificialmente. |
| 4. Por que elas ajudam? | redução estimada de perda na origem e no destino; trade-off da transferência | Variação marginal da função de perda antes/depois de cada movimento | Explicar benefício líquido previsto, não prometer viagens efetivamente salvas. |
| 5. Quão confiável é a previsão? | MAE, RMSE, viés e limites da modelagem | Validação cronológica da seção 5; MAE de 0,699 para retiradas e 0,708 para devoluções | Distinguir erro preditivo da melhora no backtest; mencionar censura em viagens realizadas e status exógeno. |

## 8.2 Decisões de produto herdadas dos experimentos

1. A tela deve priorizar risco direcional, e não uma meta genérica de 50% de ocupação: P2/P3/P7 mostraram que retirada, devolução e horário alteram o estoque desejável.
2. A recomendação deve projetar duas horas, mas descrever ao usuário somente o risco imediato e o efeito esperado da ação. O segundo período existe para evitar que a intervenção resolva o presente e agrave o próximo pico.
3. Estações inativas não são candidatas a origem ou destino. A interface deve destacá-las como indisponibilidade operacional, em vez de classificá-las como falha de rebalanceamento.
4. A confiança deve ser calibrada pela validação histórica, e não por precisão aparente de casas decimais. P13 foi escolhido porque o sinal de disponibilidade ajudou no backtest, mas ele não observa o status futuro.
5. Clima, quantis, zona e segmentação por estoque não entram na recomendação atual: foram testados isoladamente e não melhoraram o score. Essa é uma escolha deliberada de robustez, não uma omissão.

## 8.3 Próximas células desta seção

As próximas células devem calcular, no snapshot, os indicadores da pergunta 1; reproduzir os riscos de P13 para a pergunta 2; executar a política sobre esse estado para as perguntas 3 e 4; e montar uma tabela compacta de métricas de validação para a pergunta 5. Assim, cada elemento exibido na interface terá uma célula-fonte no notebook.

## 8.4 Pergunta 1 — Como está a rede agora?

Esta resposta usa exclusivamente o snapshot. As métricas abaixo descrevem o estado observado às 14:30, sem inferência sobre demanda futura.

In [210]:
snapshot_interface = snapshot_atual.copy()
snapshot_interface["ocupacao"] = snapshot_interface["available_bikes"] / snapshot_interface["capacity"]
snapshot_interface["critica_baixa"] = snapshot_interface["status"].eq("active") & snapshot_interface["ocupacao"].le(0.10)
snapshot_interface["critica_alta"] = snapshot_interface["status"].eq("active") & snapshot_interface["ocupacao"].ge(0.90)
momento_interface = pd.Timestamp(snapshot_interface["last_update"].iloc[0])

pd.Series({
    "momento observado": momento_interface,
    "bicicletas disponíveis": snapshot_interface["available_bikes"].sum(),
    "vagas disponíveis": snapshot_interface["available_docks"].sum(),
    "capacidade total": snapshot_interface["capacity"].sum(),
    "ocupação da rede (%)": round(100 * snapshot_interface["available_bikes"].sum() / snapshot_interface["capacity"].sum(), 1),
    "estações ativas": snapshot_interface["status"].eq("active").sum(),
    "estações inativas/manutenção": (~snapshot_interface["status"].eq("active")).sum(),
    "críticas baixas ativas (≤10%)": snapshot_interface["critica_baixa"].sum(),
    "críticas altas ativas (≥90%)": snapshot_interface["critica_alta"].sum(),
}, name="valor")

momento observado                2024-09-30 14:30:00
bicicletas disponíveis                           702
vagas disponíveis                                922
capacidade total                                1624
ocupação da rede (%)                          43.200
estações ativas                                   77
estações inativas/manutenção                       3
críticas baixas ativas (≤10%)                     13
críticas altas ativas (≥90%)                       9
Name: valor, dtype: object

In [211]:
resumo_zonas_interface = (
    snapshot_interface.groupby("zone", as_index=False)
    .agg(
        estações=("station_id", "size"),
        ativas=("status", lambda s: s.eq("active").sum()),
        bicicletas=("available_bikes", "sum"),
        vagas=("available_docks", "sum"),
        capacidade=("capacity", "sum"),
        críticas_baixas=("critica_baixa", "sum"),
        críticas_altas=("critica_alta", "sum"),
    )
)
resumo_zonas_interface["ocupação (%)"] = 100 * resumo_zonas_interface["bicicletas"] / resumo_zonas_interface["capacidade"]
resumo_zonas_interface.sort_values("zone").round(1)

,zone,estações,ativas,bicicletas,vagas,capacidade,críticas_baixas,críticas_altas,ocupação (%)
0,Centro,14,14,150,129,279,1,4,53.800
1,Zona Leste,14,14,99,178,277,3,1,35.700
2,Zona Norte,14,13,87,187,274,5,1,31.800
3,Zona Oeste,18,16,188,187,375,1,3,50.100
4,Zona Sul,20,20,178,241,419,3,0,42.500


In [212]:
colunas_estado = ["station_id", "name", "zone", "status", "available_bikes", "available_docks", "capacity", "ocupacao"]
estacoes_criticas_interface = snapshot_interface.loc[
    snapshot_interface["critica_baixa"] | snapshot_interface["critica_alta"], colunas_estado
].sort_values(["ocupacao", "station_id"])
estacoes_indisponiveis_interface = snapshot_interface.loc[
    ~snapshot_interface["status"].eq("active"), colunas_estado
].sort_values(["status", "station_id"])

display(estacoes_criticas_interface.assign(ocupacao_pct=lambda df: (100 * df["ocupacao"]).round(1)).drop(columns="ocupacao"))
display(estacoes_indisponiveis_interface.assign(ocupacao_pct=lambda df: (100 * df["ocupacao"]).round(1)).drop(columns="ocupacao"))

,station_id,name,zone,status,available_bikes,available_docks,capacity,ocupacao_pct
9,ST010,Mercado Municipal,Centro,active,0,12,12,0.000
32,ST033,Vila Clementino,Zona Sul,active,0,17,17,0.000
59,ST060,Jaçanã,Zona Norte,active,0,14,14,0.000
60,ST061,Mandaqui,Zona Norte,active,0,23,23,0.000
65,ST066,Parada Inglesa,Zona Norte,active,0,20,20,0.000
70,ST071,Belém,Zona Leste,active,0,23,23,0.000
74,ST075,Parque do Carmo,Zona Leste,active,0,22,22,0.000
76,ST077,São Mateus,Zona Leste,active,0,16,16,0.000
56,ST057,Freguesia do Ó,Zona Norte,active,1,22,23,4.300
30,ST031,Museu do Ipiranga,Zona Sul,active,1,21,22,4.500


,station_id,name,zone,status,available_bikes,available_docks,capacity,ocupacao_pct
46,ST047,Vila Leopoldina,Zona Oeste,inactive,5,17,22,22.700
50,ST051,Vila Sônia,Zona Oeste,maintenance,1,11,12,8.300
55,ST056,Casa Verde,Zona Norte,maintenance,3,20,23,13.000


## 8.5 Pergunta 2 — O que pode acontecer na próxima hora?

Para evitar olhar o futuro do snapshot, os perfis abaixo usam somente viagens e status com horário anterior a 30/09 às 14:30. Como o modelo é horário, a previsão começa na próxima hora cheia (15:00) e projeta também a hora seguinte para preservar a lógica de P13.

In [213]:
momento_risco = momento_interface.ceil("h")
viagens_interface = viagens.loc[(viagens["start_time"] < momento_interface) & (viagens["end_time"] < momento_interface)].copy()

def perfil_historico_interface(df, coluna_estacao, coluna_tempo, nome):
    eventos = df.assign(data=df[coluna_tempo].dt.normalize(), hora=df[coluna_tempo].dt.hour, dia_semana=df[coluna_tempo].dt.dayofweek)
    contagens = eventos.groupby([coluna_estacao, "hora", "dia_semana"]).size().rename(nome).reset_index()
    dias = eventos.groupby("dia_semana")["data"].nunique().rename("n_dias").reset_index()
    return contagens.merge(dias, on="dia_semana", how="left").assign(**{nome: lambda df: df[nome] / df["n_dias"]}).drop(columns="n_dias").rename(columns={coluna_estacao: "station_id"})

retiradas_interface = perfil_historico_interface(viagens_interface, "start_station_id", "start_time", "retiradas_previstas")
devolucoes_interface = perfil_historico_interface(viagens_interface, "end_station_id", "end_time", "devolucoes_previstas")
perfil_interface = (
    retiradas_interface.merge(devolucoes_interface, on=["station_id", "hora", "dia_semana"], how="outer")
    .fillna({"retiradas_previstas": 0.0, "devolucoes_previstas": 0.0})
)
perfil_lookup_interface = {
    (linha.station_id, linha.hora, linha.dia_semana): (linha.retiradas_previstas, linha.devolucoes_previstas)
    for linha in perfil_interface.itertuples(index=False)
}

status_interface = historico_status.loc[historico_status["timestamp"] < momento_interface].copy()
status_interface["hora"] = status_interface["timestamp"].dt.hour
status_interface["dia_semana"] = status_interface["timestamp"].dt.dayofweek
confiabilidade_lookup_interface = (
    status_interface.assign(ativo=status_interface["status"].eq("active"))
    .groupby(["station_id", "hora", "dia_semana"])["ativo"].mean().to_dict()
)
pd.Series({"viagens usadas": len(viagens_interface), "observações de status usadas": len(status_interface), "início da previsão": momento_risco})

viagens usadas                               164777
observações de status usadas                 703520
início da previsão              2024-09-30 15:00:00
dtype: object

In [214]:
def perfil_p13_interface(station_id, instante):
    retiradas, devolucoes = perfil_lookup_interface.get((station_id, instante.hour, instante.dayofweek), (0.0, 0.0))
    return (0.0 if instante.hour < 5 or instante.hour >= 22 else retiradas), devolucoes

def perda_interface(bikes, capacidade, retiradas, devolucoes):
    return max(retiradas - bikes, 0.0) + max(devolucoes - (capacidade - bikes), 0.0)

def decompor_risco_p13(estacao, bikes, instante):
    r0, a0 = perfil_p13_interface(estacao["station_id"], instante)
    perda_retirada = max(r0 - bikes, 0.0)
    perda_devolucao = max(a0 - (estacao["capacity"] - bikes), 0.0)
    bikes_1 = min(estacao["capacity"], max(0.0, bikes - r0 + a0))
    proximo = instante + pd.Timedelta(hours=1)
    r1, a1 = perfil_p13_interface(estacao["station_id"], proximo)
    confiabilidade = confiabilidade_lookup_interface.get((estacao["station_id"], proximo.hour, proximo.dayofweek), 1.0)
    perda_2h = perda_retirada + perda_devolucao + confiabilidade * perda_interface(bikes_1, estacao["capacity"], r1, a1)
    return perda_retirada, perda_devolucao, perda_2h

riscos_interface = snapshot_interface.loc[snapshot_interface["status"].eq("active")].copy()
riscos_interface[["risco_retirada_1h", "risco_devolucao_1h", "risco_total_2h"]] = riscos_interface.apply(
    lambda linha: pd.Series(decompor_risco_p13(linha, linha["available_bikes"], momento_risco)), axis=1
)
riscos_interface["risco_imediato"] = riscos_interface["risco_retirada_1h"] + riscos_interface["risco_devolucao_1h"]
riscos_interface.sort_values(["risco_total_2h", "station_id"], ascending=[False, True])[
    ["station_id", "name", "zone", "available_bikes", "available_docks", "risco_retirada_1h", "risco_devolucao_1h", "risco_total_2h"]
].head(15).round(3)

,station_id,name,zone,available_bikes,available_docks,risco_retirada_1h,risco_devolucao_1h,risco_total_2h
42,ST043,Parque Villa-Lobos,Zona Oeste,1,21,1.714,0.000,4.000
5,ST006,Praça da Liberdade,Centro,18,0,0.000,1.357,3.643
6,ST007,Largo do Arouche,Centro,21,0,0.000,1.500,3.610
9,ST010,Mercado Municipal,Centro,0,12,1.786,0.000,2.786
3,ST004,Estação Luz,Centro,20,2,0.000,0.857,2.714
74,ST075,Parque do Carmo,Zona Leste,0,22,1.143,0.000,2.500
10,ST011,Praça Roosevelt,Centro,15,1,0.000,1.429,2.143
30,ST031,Museu do Ipiranga,Zona Sul,1,21,0.429,0.000,1.929
70,ST071,Belém,Zona Leste,0,23,0.429,0.000,1.615
32,ST033,Vila Clementino,Zona Sul,0,17,0.786,0.000,1.445


## 8.6 Perguntas 3 e 4 — Quais ações são recomendadas e por que ajudam?

A célula seguinte executa uma versão explicável de P13 sobre o snapshot: em cada uma das três rodadas, ela escolhe a quantidade, origem e destino com maior ganho marginal previsto no risco de duas horas. O estado é atualizado após cada escolha; por isso, a ordem exibida é parte da recomendação.

In [218]:
def risco_p13_interface(estacao, bikes, instante):
    return decompor_risco_p13(estacao, bikes, instante)[2]

estado_p13_interface = {linha["station_id"]: linha.to_dict() for _, linha in riscos_interface.iterrows()}
detalhes_movimentos_interface = []
for ordem in range(1, 4):
    melhor = None
    for quantidade in range(1, 11):
        origens, destinos = [], []
        for station_id, estacao in estado_p13_interface.items():
            bikes = estacao["available_bikes"]
            risco_atual = risco_p13_interface(estacao, bikes, momento_risco)
            if bikes >= quantidade:
                origens.append((risco_p13_interface(estacao, bikes - quantidade, momento_risco) - risco_atual, station_id))
            if estacao["available_docks"] >= quantidade:
                destinos.append((risco_atual - risco_p13_interface(estacao, bikes + quantidade, momento_risco), station_id))
        origens.sort()
        destinos.sort(reverse=True)
        for custo_origem, origem in origens[:5]:
            for beneficio_destino, destino in destinos[:5]:
                if origem != destino:
                    candidato = (beneficio_destino - custo_origem, quantidade, origem, destino, custo_origem, beneficio_destino)
                    if melhor is None or candidato > melhor:
                        melhor = candidato
    if melhor is None or melhor[0] <= 1e-9:
        break
    ganho, quantidade, origem, destino, custo_origem, beneficio_destino = melhor
    origem_estado, destino_estado = estado_p13_interface[origem], estado_p13_interface[destino]
    detalhes_movimentos_interface.append({
        "ordem": ordem, "origem": origem, "destino": destino, "quantidade": quantidade,
        "variacao_risco_origem": custo_origem, "beneficio_previsto_destino": beneficio_destino, "ganho_liquido_previsto": ganho,
    })
    origem_estado["available_bikes"] -= quantidade
    origem_estado["available_docks"] += quantidade
    destino_estado["available_bikes"] += quantidade
    destino_estado["available_docks"] -= quantidade

movimentos_interface = pd.DataFrame(detalhes_movimentos_interface)
movimentos_interface

,ordem,origem,destino,quantidade,variacao_risco_origem,beneficio_previsto_destino,ganho_liquido_previsto
0,1,ST006,ST043,10,-3.643,4.000,7.643
1,2,ST007,ST010,10,-3.610,2.786,6.396
2,3,ST004,ST075,10,-2.714,2.500,5.214


In [219]:
nomes_interface = snapshot_interface.set_index("station_id")[["name", "zone"]]
if movimentos_interface.empty:
    print("Nenhum movimento apresentou ganho líquido previsto positivo.")
else:
    recomendacoes_interface = movimentos_interface.copy()
    recomendacoes_interface = recomendacoes_interface.join(nomes_interface.add_prefix("origem_"), on="origem")
    recomendacoes_interface = recomendacoes_interface.join(nomes_interface.add_prefix("destino_"), on="destino")
    def explicar_movimento(linha):
        efeito_origem = (
            f"reduz o risco na origem em {-linha['variacao_risco_origem']:.2f}"
            if linha["variacao_risco_origem"] <= 0
            else f"aumenta o risco na origem em {linha['variacao_risco_origem']:.2f}"
        )
        return (f"Enviar {int(linha['quantidade'])} bicicleta(s) de {linha['origem']} para {linha['destino']}: "
                f"reduz o risco no destino em {linha['beneficio_previsto_destino']:.2f} e {efeito_origem}.")

    recomendacoes_interface["justificativa"] = recomendacoes_interface.apply(explicar_movimento, axis=1)
    display(recomendacoes_interface[[
        "ordem", "origem", "origem_name", "origem_zone", "destino", "destino_name", "destino_zone", "quantidade",
        "variacao_risco_origem", "beneficio_previsto_destino", "ganho_liquido_previsto", "justificativa",
    ]].round({"variacao_risco_origem": 3, "beneficio_previsto_destino": 3, "ganho_liquido_previsto": 3}))

,ordem,origem,origem_name,origem_zone,destino,destino_name,destino_zone,quantidade,variacao_risco_origem,beneficio_previsto_destino,ganho_liquido_previsto,justificativa
0,1,ST006,Praça da Liberdade,Centro,ST043,Parque Villa-Lobos,Zona Oeste,10,-3.643,4.000,7.643,Enviar 10 bicicleta(s) de ST006 para ST043: re...
1,2,ST007,Largo do Arouche,Centro,ST010,Mercado Municipal,Centro,10,-3.610,2.786,6.396,Enviar 10 bicicleta(s) de ST007 para ST010: re...
2,3,ST004,Estação Luz,Centro,ST075,Parque do Carmo,Zona Leste,10,-2.714,2.500,5.214,Enviar 10 bicicleta(s) de ST004 para ST075: re...


## 8.7 Pergunta 5 — Quão confiável é a previsão?

A confiança não é uma probabilidade de acerto da recomendação. Ela é comunicada por desempenho fora da amostra do perfil de demanda e por limitações explícitas: `viagens.csv` observa somente viagens realizadas, e a disponibilidade futura das estações permanece incerta.

In [220]:
metricas_interface = pd.DataFrame({
    "retiradas": metricas_previsao(validacao["retiradas"], validacao["retiradas_previstas"]),
    "devoluções": metricas_previsao(validacao["devolucoes"], validacao["devolucoes_previstas"]),
}).T.round(3)

limites_interface = pd.DataFrame({
    "aspecto": ["Dados observados", "Demanda", "Status", "Uso do backtest"],
    "como comunicar": [
        "A ocupação e o status exibidos são observados no snapshot.",
        "A demanda é uma média histórica por estação, hora e dia da semana; viagens frustradas não aparecem no histórico.",
        "P13 reduz o risco futuro pela frequência histórica de atividade, mas não conhece indisponibilidades futuras.",
        "O ganho de P13 em outubro é validação retrospectiva; não é mostrado como previsão do snapshot.",
    ],
})
display(metricas_interface)
display(limites_interface)

,MAE,viés,RMSE
retiradas,0.699,0.017,1.061
devoluções,0.708,0.017,1.125


,aspecto,como comunicar
0,Dados observados,A ocupação e o status exibidos são observados ...
1,Demanda,"A demanda é uma média histórica por estação, h..."
2,Status,P13 reduz o risco futuro pela frequência histó...
3,Uso do backtest,O ganho de P13 em outubro é validação retrospe...
